In [ ]:
NOTEBOOK_TITLE = '005: DueCare Glossary and Reading Map'
from IPython.display import HTML, display
display(HTML(
    '<div style="background:linear-gradient(135deg,#1e3a8a 0%,#4c78a8 100%);'
    'color:white;padding:20px 24px;border-radius:8px;margin:8px 0;'
    'font-family:system-ui,-apple-system,sans-serif">'
    '<div style="font-size:10px;font-weight:600;letter-spacing:0.14em;'
    'opacity:0.8;text-transform:uppercase">DueCare - Orientation Navigation Surface</div>'
    f'<div style="font-size:22px;font-weight:700;margin:6px 0 4px 0">'
    f'{NOTEBOOK_TITLE}</div>'
    '<div style="font-size:13px;opacity:0.92;line-height:1.5">One place to look up what a label means and which notebook demonstrates it. 41 anchored terms in 8 scannable groups, plus 5 reading paths tied to each term.</div>'
    '</div>'
))


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    raise RuntimeError(
        f'DueCare version {duecare.core.__version__} installed but this notebook was written against 0.1.0. Restart the runtime, rerun the install cell, and confirm the wheel dataset is current.'
    )


# 005: DueCare Glossary and Reading Map

This notebook defines the project vocabulary and maps each term back to the notebook that demonstrates it. Open it whenever a label, route, or deployment-surface name in the suite needs to be pinned down before clicking deeper.

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">Pinned DueCare packages only; no domain-specific data or external APIs.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">Glossary tables, a route map derived from the 000 phase registry, and a live registry proof from the installed packages.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">Kaggle CPU kernel with internet enabled. No GPU. No API keys.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">Under 2 minutes end-to-end.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Background and Package Setup section. Previous: <a href="https://www.kaggle.com/code/taylorsamarel/duecare-000-index">000 Index</a>. Next: <a href="https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes">010 Quickstart</a>. Section close: <a href="https://www.kaggle.com/code/taylorsamarel/099-duecare-orientation-setup-conclusion">099 Conclusion</a>.</td></tr>
  </tbody>
</table>


### How 005 differs from 000 Index

[**000 Index**](https://www.kaggle.com/code/taylorsamarel/duecare-000-index) is the suite-map surface. It lists every section, every notebook, the section coverage table, and the recommended reading routes. If you want to know **where to click next**, open 000.

005 is the vocabulary surface. It does not re-list the sections or the suite map. It answers a different question: **what does this name mean**, and **which notebook is the canonical demonstration**. Term first, notebook second. Eight thematic groups scan top to bottom: project framing, notation, data and grading, prompt engineering, evaluation, orchestration and training, export and deployment, safety and domain.

**Privacy is non-negotiable** is the project's load-bearing invariant: raw migrant-worker case data never leaves the deployment boundary. Every term on this page maps back to a notebook that either enforces that boundary (Anonymizer, AgentSupervisor, the on-device GGUF runtime) or proves the model still meets its safety rubric while doing so. The named partners who would deploy DueCare against this guarantee — Polaris, IJM, ECPAT, POEA, BP2MI, HRD Nepal, IOM — appear in the Solution Surfaces section.

This notebook is CPU-only by design. Gemma 4 inference starts in [100 Gemma Exploration](https://www.kaggle.com/code/taylorsamarel/duecare-real-gemma-4-on-50-trafficking-prompts) on a T4 GPU.


## At a glance


In [ ]:
from IPython.display import HTML, display

_PALETTE = {
    "primary": "#4c78a8", "success": "#10b981", "info": "#3b82f6",
    "warning": "#f59e0b", "muted": "#6b7280", "danger": "#ef4444",
    "bg_primary": "#eff6ff", "bg_success": "#ecfdf5",
    "bg_info": "#eff6ff", "bg_warning": "#fffbeb", "bg_danger": "#fef2f2",
}

def _stat_card(value, label, sub, kind="primary"):
    accent = _PALETTE[kind]
    bg = _PALETTE.get(f"bg_{kind}", _PALETTE["bg_info"])
    return (
        f'<div style="display:inline-block;vertical-align:top;width:22%;'
        f'margin:4px 1%;padding:14px 16px;background:{bg};'
        f'border-left:5px solid {accent};border-radius:4px;'
        f'font-family:system-ui,-apple-system,sans-serif">'
        f'<div style="font-size:11px;font-weight:600;color:{accent};'
        f'text-transform:uppercase;letter-spacing:0.04em">{label}</div>'
        f'<div style="font-size:26px;font-weight:700;color:#1f2937;margin:4px 0 0 0">{value}</div>'
        f'<div style="font-size:12px;color:{_PALETTE["muted"]};margin-top:2px">{sub}</div>'
        f'</div>'
    )

n_terms = 41
cards = [
    _stat_card(f"{n_terms}",  "glossary terms",     "anchored to notebooks",    "primary"),
    _stat_card("8",                     "topic groups",       "framing -> deployment",    "info"),
    _stat_card("5",  "reading paths",      "judge / adopter / depth",  "success"),
    _stat_card("7",                     "rubric dimensions",  "5-grade + 6 weighted",     "warning"),
]
display(HTML('<div style="margin:8px 0">' + ''.join(cards) + '</div>'))


## Reading paths

Paths through the suite for different reader goals. These routes complement the 000 Index and every notebook ID below is clickable.

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 8px 10px; text-align: left; width: 18%;">Goal</th>
      <th style="padding: 8px 10px; text-align: left; width: 42%;">Path</th>
      <th style="padding: 8px 10px; text-align: left; width: 40%;">Why this path</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Judge fast path</b></td><td style="padding: 6px 10px; font-family: monospace; font-size: 0.92em;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-000-index"><b>000</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes"><b>010</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard"><b>600</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/610-duecare-submission-walkthrough"><b>610</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion"><b>899</b></a></td><td style="padding: 6px 10px;">Shortest competition path: prove the install, open the headline dashboard, then close on the capstone and solution surfaces.</td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Technical proof path</b></td><td style="padding: 6px 10px; font-family: monospace; font-size: 0.92em;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-real-gemma-4-on-50-trafficking-prompts"><b>100</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/duecare-cross-domain-proof"><b>200</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/duecare-500-agent-swarm-deep-dive"><b>500</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune"><b>530</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/540-duecare-fine-tune-delta-visualizer"><b>540</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard"><b>600</b></a></td><td style="padding: 6px 10px;">Best route for a judge checking baseline behavior, cross-domain generalization, orchestration, fine-tuning, and the public charts.</td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Adopter path</b></td><td style="padding: 6px 10px; font-family: monospace; font-size: 0.92em;"><a href="https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes"><b>010</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/duecare-cross-domain-proof"><b>200</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour"><b>620</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough"><b>650</b></a></td><td style="padding: 6px 10px;">Fastest route for an NGO, regulator, or engineering team deciding whether the system is reusable in their own stack.</td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Interactive demos</b></td><td style="padding: 6px 10px; font-family: monospace; font-size: 0.92em;"><a href="https://www.kaggle.com/code/taylorsamarel/150-duecare-free-form-gemma-playground"><b>150</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/155-duecare-tool-calling-playground"><b>155</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground"><b>160</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/180-duecare-multimodal-document-inspector"><b>180</b></a></td><td style="padding: 6px 10px;">Type prompts, trigger tool-calling, upload images, and inspect multimodal recruitment documents.</td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Evaluation depth</b></td><td style="padding: 6px 10px; font-family: monospace; font-size: 0.92em;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-cross-domain-proof"><b>200</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/duecare-500-agent-swarm-deep-dive"><b>500</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune"><b>530</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard"><b>600</b></a> &rarr; <a href="https://www.kaggle.com/code/taylorsamarel/610-duecare-submission-walkthrough"><b>610</b></a></td><td style="padding: 6px 10px;">One compact route from proof of generalization through the improvement spine into the public-facing results and capstone walkthrough.</td></tr>
  </tbody>
</table>

## Glossary: Project framing

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 8px 10px; text-align: left; width: 22%;">Term</th>
      <th style="padding: 8px 10px; text-align: left; width: 54%;">Meaning</th>
      <th style="padding: 8px 10px; text-align: left; width: 24%;">Primary notebook</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>DueCare</b></td><td style="padding: 6px 10px;">The project and notebook suite: an on-device LLM safety harness for sensitive domains, named for the duty-of-care standard.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-000-index">000 Index</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Duty of care</b></td><td style="padding: 6px 10px;">The common-law standard codified in California Civil Code section 1714(a) and applied by a California jury in March 2026 when it found Meta and Google negligent for defective platform design.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-000-index">000 Index</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Composite character</b></td><td style="padding: 6px 10px;">A named but fictional worker (Maria, Ramesh) used for storytelling without exposing a real person.</td><td style="padding: 6px 10px; white-space: nowrap;"><span style="color: #6a737d;">&mdash;</span></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Privacy as invariant</b></td><td style="padding: 6px 10px;">Raw case data never leaves the deployment boundary; every solution surface is designed to meet this before any accuracy claim.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes">010 Quickstart</a></td></tr>
  </tbody>
</table>

## Glossary: Notation and structure

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 8px 10px; text-align: left; width: 22%;">Term</th>
      <th style="padding: 8px 10px; text-align: left; width: 54%;">Meaning</th>
      <th style="padding: 8px 10px; text-align: left; width: 24%;">Primary notebook</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>NNN prefix</b></td><td style="padding: 6px 10px;">Three-digit zero-padded notebook ID. The first digit names the functional band (100s exploration, 200s comparison, 300s adversarial), not the reading order. Step of 10 between siblings leaves room for insertion.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-000-index">000 Index</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>N99 conclusion</b></td><td style="padding: 6px 10px;">The section-closing notebook at the top of each 100-slot band (099, 199 through 899). Recap, key points, and the next-section handoff.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/099-duecare-orientation-setup-conclusion">099 Conclusion</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Section map</b></td><td style="padding: 6px 10px;">The 16-section reading order shown in 000 Index spans orientation, exploration, jailbreak research, baseline text evaluation, comparisons, advanced evaluation, model improvement, results, solution surfaces, deployment applications, and the explicit planned image-side sections.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-000-index">000 Index</a></td></tr>
  </tbody>
</table>

## Glossary: Data and grading

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 8px 10px; text-align: left; width: 22%;">Term</th>
      <th style="padding: 8px 10px; text-align: left; width: 54%;">Meaning</th>
      <th style="padding: 8px 10px; text-align: left; width: 24%;">Primary notebook</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Domain pack</b></td><td style="padding: 6px 10px;">A packaged safety domain with prompts, rubric criteria, taxonomy, evidence, and deployment context.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-cross-domain-proof">200 Cross-Domain Proof</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Graded response</b></td><td style="padding: 6px 10px;">A reference answer on a quality ladder (worst, neutral, good, best) for the same prompt.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/00a-duecare-prompt-prioritizer-data-pipeline">110 Prompt Prioritizer</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Rubric</b></td><td style="padding: 6px 10px;">The criterion set used to judge whether a response is safe, complete, accurate, and actionable.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-430-rubric-evaluation">430 Rubric Evaluation</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Anchored grading</b></td><td style="padding: 6px 10px;">Scoring by comparing against hand-written best and worst references, not against a free-floating judge.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-250-comparative-grading">250 Comparative Grading</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>LLM judge</b></td><td style="padding: 6px 10px;">A model used to score another model's answer across dimensions rather than answering the task itself.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading">410 LLM Judge</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Contextual judge</b></td><td style="padding: 6px 10px;">A judge that evaluates the response relative to the prompt context, not in isolation.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-contextual-judge">450 Contextual Worst-Response</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Failure taxonomy</b></td><td style="padding: 6px 10px;">The categorized description of where the model fails: refusal, legality, actionability, empathy, and more.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-per-prompt-rubric-generator">440 Per-Prompt Rubric Generator</a></td></tr>
  </tbody>
</table>

## Glossary: Prompt engineering

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 8px 10px; text-align: left; width: 22%;">Term</th>
      <th style="padding: 8px 10px; text-align: left; width: 54%;">Meaning</th>
      <th style="padding: 8px 10px; text-align: left; width: 24%;">Primary notebook</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Prompt prioritizer</b></td><td style="padding: 6px 10px;">The curator that selects the highest-value prompts from the seed corpus before any scored evaluation.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/00a-duecare-prompt-prioritizer-data-pipeline">110 Prompt Prioritizer</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Prompt remixer</b></td><td style="padding: 6px 10px;">A generator that expands a curated prompt into alternative phrasings and adversarial variants.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-prompt-remixer">120 Prompt Remixer</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Prompt factory</b></td><td style="padding: 6px 10px;">The scaled pipeline that generates, validates, and ranks large volumes of test prompts by victim impact.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-310-prompt-factory">310 Prompt Factory</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Per-prompt rubric</b></td><td style="padding: 6px 10px;">A rubric synthesized for a single prompt so scoring matches the specific failure mode that prompt targets.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-per-prompt-rubric-generator">440 Per-Prompt Rubric Generator</a></td></tr>
  </tbody>
</table>

## Glossary: Evaluation

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 8px 10px; text-align: left; width: 22%;">Term</th>
      <th style="padding: 8px 10px; text-align: left; width: 54%;">Meaning</th>
      <th style="padding: 8px 10px; text-align: left; width: 24%;">Primary notebook</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Cross-domain proof</b></td><td style="padding: 6px 10px;">The claim that the same harness works across trafficking, tax evasion, and financial crime.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-cross-domain-proof">200 Cross-Domain Proof</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>RAG</b></td><td style="padding: 6px 10px;">Retrieval-augmented generation: inject domain evidence or rubric context before generating a response.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-260-rag-comparison">260 RAG Comparison</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Function calling</b></td><td style="padding: 6px 10px;">Structured tool invocation by the model using native function-call syntax rather than plain-text instructions.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-400-function-calling-multimodal">400 Function Calling and Multimodal</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Multimodal</b></td><td style="padding: 6px 10px;">Using both text and image inputs, such as document photos, in a single evaluation.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground">160 Image Processing Playground</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Safety line</b></td><td style="padding: 6px 10px;">The measured threshold where Gemma starts to produce unsafe or uncensored output under adversarial pressure.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-finding-gemma-4-safety-line">320 Finding Gemma 4 Safety Line</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Playground</b></td><td style="padding: 6px 10px;">An interactive notebook with an ipywidgets textarea or file upload so a reader can type any prompt or upload any image and see Gemma respond live.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/150-duecare-free-form-gemma-playground">150 Free Form Gemma Playground</a></td></tr>
  </tbody>
</table>

## Glossary: Orchestration and training

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 8px 10px; text-align: left; width: 22%;">Term</th>
      <th style="padding: 8px 10px; text-align: left; width: 54%;">Meaning</th>
      <th style="padding: 8px 10px; text-align: left; width: 24%;">Primary notebook</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Agent swarm</b></td><td style="padding: 6px 10px;">The set of autonomous agents coordinated to probe, grade, summarize, and publish results.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-500-agent-swarm-deep-dive">500 Agent Swarm Deep Dive</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Coordinator</b></td><td style="padding: 6px 10px;">The supervisory control layer that routes tasks, often using Gemma 4 native function calling.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-500-agent-swarm-deep-dive">500 Agent Swarm Deep Dive</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Curriculum</b></td><td style="padding: 6px 10px;">The training dataset assembled for fine-tuning, usually from graded prompts and contrastive examples.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-520-phase3-curriculum-builder">520 Phase 3 Curriculum Builder</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Unsloth</b></td><td style="padding: 6px 10px;">The fine-tuning stack used here for efficient Gemma LoRA training and export.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune">530 Phase 3 Unsloth Fine-tune</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>LoRA</b></td><td style="padding: 6px 10px;">Low-Rank Adaptation: a parameter-efficient fine-tuning method that trains small adapters instead of the full model.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune">530 Phase 3 Unsloth Fine-tune</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>4-bit quantization</b></td><td style="padding: 6px 10px;">Loading a model in 4-bit weights (nf4 via bitsandbytes) so a 9B Gemma fits on a single Kaggle T4.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/150-duecare-free-form-gemma-playground">150 Free Form Gemma Playground</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>SuperGemma</b></td><td style="padding: 6px 10px;">Internal shorthand for the DueCare-fine-tuned Gemma 4 variant produced by the 530 Unsloth path. Not a public product name.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune">530 Phase 3 Unsloth Fine-tune</a></td></tr>
  </tbody>
</table>

## Glossary: Export and deployment

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 8px 10px; text-align: left; width: 22%;">Term</th>
      <th style="padding: 8px 10px; text-align: left; width: 54%;">Meaning</th>
      <th style="padding: 8px 10px; text-align: left; width: 24%;">Primary notebook</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>GGUF</b></td><td style="padding: 6px 10px;">A quantized model format used for local inference in llama.cpp.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune">530 Phase 3 Unsloth Fine-tune</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>LiteRT</b></td><td style="padding: 6px 10px;">A mobile-oriented runtime path for private on-device deployment.</td><td style="padding: 6px 10px; white-space: nowrap;"><span style="color: #6a737d;">&mdash;</span></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Enterprise surface</b></td><td style="padding: 6px 10px;">Deployment shape 1 of 4: platform-side content moderation, compliance monitoring, NGO review dashboards.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard">600 Results Dashboard</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Client surface</b></td><td style="padding: 6px 10px;">Deployment shape 2 of 4: on-device verification on a single migrant worker's laptop or phone.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes">010 Quickstart</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>NGO API surface</b></td><td style="padding: 6px 10px;">Deployment shape 3 of 4: a generalized public API endpoint backed by the agent swarm orchestration layer.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour">620 Demo API Endpoint Tour</a></td></tr>
  </tbody>
</table>

## Glossary: Safety and domain

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 8px 10px; text-align: left; width: 22%;">Term</th>
      <th style="padding: 8px 10px; text-align: left; width: 54%;">Meaning</th>
      <th style="padding: 8px 10px; text-align: left; width: 24%;">Primary notebook</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Case analysis surface</b></td><td style="padding: 6px 10px;">Deployment shape 4 of 4: tool-backed, multimodal, contextual case review for victims and advocates.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough">650 Custom Domain Walkthrough</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>PII gate</b></td><td style="padding: 6px 10px;">The anonymization boundary that strips or generalizes sensitive personal information before downstream use.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/610-duecare-submission-walkthrough">610 Submission Walkthrough</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Migration corridor</b></td><td style="padding: 6px 10px;">A worker movement pattern between countries, such as Nepal to UAE or Philippines to Hong Kong.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-cross-domain-proof">200 Cross-Domain Proof</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>ILO indicators</b></td><td style="padding: 6px 10px;">Forced-labor and trafficking indicators defined by the International Labour Organization.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/duecare-cross-domain-proof">200 Cross-Domain Proof</a></td></tr>
    <tr><td style="padding: 6px 10px; white-space: nowrap;"><b>Named partners</b></td><td style="padding: 6px 10px;">Public organizations the project references as plausible real-world deployers: Polaris Project, IJM, ECPAT, POEA, BP2MI, HRD Nepal.</td><td style="padding: 6px 10px; white-space: nowrap;"><a href="https://www.kaggle.com/code/taylorsamarel/610-duecare-submission-walkthrough">610 Submission Walkthrough</a></td></tr>
  </tbody>
</table>

## Registries on disk

The glossary is not just prose. The cell below imports the shipped DueCare packages and prints the live registries so you can confirm the domain packs, tasks, agents, and model adapters actually exist. A healthy run should show nonzero model, domain, task, and agent counts; the assertions below fail loudly if any of them are empty.


In [ ]:
from duecare.agents import agent_registry
from duecare.domains import domain_registry, register_discovered
from duecare.models import model_registry
from duecare.tasks import task_registry

register_discovered()

model_count = len(model_registry)
domain_count = len(domain_registry)
task_count = len(task_registry)
agent_count = len(agent_registry)

print(f'Model adapters:   {model_count}')
print(f'Domain packs:     {domain_count}')
print(f'Capability tests: {task_count}')
print(f'Agents:           {agent_count}')
print()
print('Domains:', domain_registry.all_ids())
print('Tasks:  ', task_registry.all_ids()[:8])
print('Agents: ', agent_registry.all_ids()[:8])

assert model_count > 0, 'Model registry is empty; the duecare-llm-models install likely failed.'
assert domain_count > 0, 'Domain registry is empty; run register_discovered() and confirm the domains wheel is installed.'
assert task_count > 0, 'Task registry is empty; the duecare-llm-tasks install likely failed.'
assert agent_count > 0, 'Agent registry is empty; the duecare-llm-agents install likely failed.'


## Troubleshooting

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 36%;">Symptom</th>
      <th style="padding: 6px 10px; text-align: left; width: 64%;">Resolution</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;">Pinned install fails and raises a <code>RuntimeError</code> from the fallback.</td><td style="padding: 6px 10px;">Attach the <code>taylorsamarel/duecare-llm-wheels</code> dataset from the Kaggle sidebar and rerun the install cell.</td></tr>
    <tr><td style="padding: 6px 10px;"><code>RuntimeError</code> about a version mismatch.</td><td style="padding: 6px 10px;">Restart the runtime (Run &rarr; Factory reset and clear output), then rerun the install cell so the new version fully replaces the cached one.</td></tr>
    <tr><td style="padding: 6px 10px;">An <code>assert</code> fires about an empty registry.</td><td style="padding: 6px 10px;">Confirm all 8 packages installed by running <code>!pip list | grep duecare</code>. If any are missing, rerun the install cell.</td></tr>
    <tr><td style="padding: 6px 10px;">Glossary link returns a Kaggle 404.</td><td style="padding: 6px 10px;">That notebook likely uses a legacy public slug. Rebuild this glossary from <code>scripts/build_notebook_005_glossary.py</code> and confirm the target notebook is represented in <code>scripts/_public_slugs.py</code>.</td></tr>
  </tbody>
</table>


## Next

- **Continue the section:** [010 Quickstart](https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes) runs the smallest DueCare smoke test end-to-end.
- **Close the orientation section:** [099 Background and Package Setup Conclusion](https://www.kaggle.com/code/taylorsamarel/099-duecare-orientation-setup-conclusion).
- **Back to navigation:** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).


In [ ]:
print('Glossary handoff >>> 010 Quickstart https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes | 099 Conclusion https://www.kaggle.com/code/taylorsamarel/099-duecare-orientation-setup-conclusion')
